In [1]:
import torch
import transformers
from transformers import AutoConfig, AutoProcessor, Qwen3VLForConditionalGeneration
from PIL import Image

BASE_MODEL_ID = "Qwen3VL-2B-Medmaxv2"

# ==========================================
# PATCH 1: Fix Model Config (rope_scaling)
# ==========================================

model = Qwen3VLForConditionalGeneration.from_pretrained(
    BASE_MODEL_ID,
    device_map="auto",
    trust_remote_code=True,
)

model.eval()

/opt/homebrew/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|███████████████████████| 625/625 [00:01<00:00, 518.68it/s]


Qwen3VLForConditionalGeneration(
  (model): Qwen3VLModel(
    (visual): Qwen3VLVisionModel(
      (patch_embed): Qwen3VLVisionPatchEmbed(
        (proj): Conv3d(3, 1024, kernel_size=(2, 16, 16), stride=(2, 16, 16))
      )
      (pos_embed): Embedding(2304, 1024)
      (rotary_pos_emb): Qwen3VLVisionRotaryEmbedding()
      (blocks): ModuleList(
        (0-23): 24 x Qwen3VLVisionBlock(
          (norm1): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
          (norm2): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
          (attn): Qwen3VLVisionAttention(
            (qkv): Linear(in_features=1024, out_features=3072, bias=True)
            (proj): Linear(in_features=1024, out_features=1024, bias=True)
          )
          (mlp): Qwen3VLVisionMLP(
            (linear_fc1): Linear(in_features=1024, out_features=4096, bias=True)
            (linear_fc2): Linear(in_features=4096, out_features=1024, bias=True)
            (act_fn): GELUTanh()
          )
        )
      )
 

In [2]:
processor = AutoProcessor.from_pretrained(
    BASE_MODEL_ID, 
    trust_remote_code=True,
)

objc[59772]: Class AVFFrameReceiver is implemented in both /opt/homebrew/lib/python3.11/site-packages/av/.dylibs/libavdevice.62.1.100.dylib (0x10f2543a8) and /opt/homebrew/lib/python3.11/site-packages/cv2/.dylibs/libavdevice.61.3.100.dylib (0x12f5003a8). This may cause spurious casting failures and mysterious crashes. One of the duplicates must be removed or renamed.
objc[59772]: Class AVFAudioReceiver is implemented in both /opt/homebrew/lib/python3.11/site-packages/av/.dylibs/libavdevice.62.1.100.dylib (0x10f2543f8) and /opt/homebrew/lib/python3.11/site-packages/cv2/.dylibs/libavdevice.61.3.100.dylib (0x12f5003f8). This may cause spurious casting failures and mysterious crashes. One of the duplicates must be removed or renamed.


In [3]:

import re
import time


def load_direct_image(path: str) -> Image.Image:
    with Image.open(path) as raw:
        img = raw.convert("RGB")
        return img

# 5. Formulate the Query
prompt = """Choose the correct option for the question

Question: 
Examine the mammogram image shown above. Which of the following findings is most evident?

Options
A. Well-circumscribed round mass with benign features
B. Clustered microcalcifications within an area of irregular density
C. Fat-containing lesion consistent with lipoma
D. Diffuse bilateral breast edema

Provide answer in detailed reasoning and final output

"""

img = load_direct_image("MM-1-a.png")

messages = [
    {
        "role": "system",
        "content": "You are an expert medical AI. You must deeply analyze the question and provide the final answer."
    },
    {
        "role": "user",
        "content": [
            {"type": "image"},
            {"type": "text", "text": prompt}
        ]
    }
]


In [4]:

stop_token_id = processor.tokenizer.convert_tokens_to_ids("<|im_end|>")

with torch.inference_mode():
    text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = processor(
        text=text,
        images=[img],
        return_tensors="pt",
    ).to("mps")

    # --- Start Benchmarking ---
    torch.mps.synchronize()  # Wait for all previous GPU tasks to finish
    start_time = time.perf_counter()

    generated_ids = model.generate(
        **inputs,
        max_new_tokens=1024,      # tight control (prevents drift)
        do_sample=False,        # deterministic output
        pad_token_id=processor.tokenizer.pad_token_id,
        eos_token_id=stop_token_id
    )
    
    torch.mps.synchronize()  # Wait for generation to completely finish
    end_time = time.perf_counter()
    # --- End Benchmarking ---

    # Isolate only the newly generated tokens
    new_tokens = generated_ids[:, inputs.input_ids.shape[1]:]

    output_text = processor.batch_decode(
        new_tokens,
        skip_special_tokens=True
    )[0]

    # Calculate Speed Metrics
    num_generated_tokens = new_tokens.shape[1]
    generation_time = end_time - start_time
    tokens_per_second = num_generated_tokens / generation_time


In [5]:

# Print Results
print("=== Output ===")
print(output_text.strip())
print("\n=== Performance Metrics ===")
print(f"Total New Tokens:  {num_generated_tokens}")
print(f"Generation Time:   {generation_time:.4f} seconds")
print(f"Token Speed:       {tokens_per_second:.2f} tokens/sec")

=== Output ===
So, let's analyze this mammogram image. First, I need to recall the key features of each option.

Option A: Well-circumscribed round mass with benign features. In mammograms, a well-circumscribed mass is often benign, but the image here doesn't show a clear mass; it's more about the density and calcifications.

Option B: Clustered microcalcifications within an area of irregular density. Microcalcifications are small calcium deposits, often seen in breast cancer, but they can also be benign. In this image, there are areas with irregular density, and microcalcifications are a key feature in some breast conditions, especially when associated with malignancy.

Option C: Fat-containing lesion consistent with lipoma. Lipomas are benign fatty tumors, and they typically appear as well-defined, homogeneous masses. The image here doesn't show a fat-containing lesion; it's more about the density and calcifications.

Option D: Diffuse bilateral breast edema. Edema is swelling due to

In [1]:
!pip install --upgrade mlx-vlm

INFO: pip is looking at multiple versions of fastapi to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 6.7 MB/s  0:00:006.8 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 9.8 MB/s  0:00:009.1 MB/s eta 0:00:01
  Attempting uninstall: llguidance
    Found existing installation: llguidance 1.3.0
    Uninstalling llguidance-1.3.0:
      Successfully uninstalled llguidance-1.3.0
  Attempting uninstall: starlette
    Found existing installation: starlette 0.52.1
    Uninstalling starlette-0.52.1:
      Successfully uninstalled starlette-0.52.1
  Attempting uninstall: fastapi
    Found existing installation: fastapi 0.129.0
    Uninstalling fastapi-0.129.0:
      Successfully uninstalled fastapi-0.129.0
  Attempting uninstall: mlx-vlm
    Found existing installation: mlx-vlm 0.5.0
    Uninstalling mlx-vlm-0.5.0:
      Successfully uninstalled mlx-vlm-0.5.0
   ━━━━━━━━━━━━━━

In [3]:
import json

# 1. Define paths
local_hf_path = "./Qwen3VL-2B-Medmaxv2"
config_path = f"{local_hf_path}/config.json"

print("Step 1: Patching config.json for MLX compatibility and 3D Vision Math...")

with open(config_path, "r") as f:
    config_data = json.load(f)

# A. Fix the strict MLX name requirement so it compiles
if "vision_config" in config_data:
    config_data["vision_config"]["model_type"] = "qwen3_vl"

# B. Inject the strict 3D Spatial Math (mRoPE) so the model retains its sight
if "text_config" in config_data:
    if "rope_theta" not in config_data["text_config"]:
        config_data["text_config"]["rope_theta"] = 1000000.0
    
    # This prevents the model from turning 2D images into 1D noise
    config_data["text_config"]["rope_scaling"] = {
        "rope_type": "default",
        "mrope_section": [24, 20, 20]
    }

with open(config_path, "w") as f:
    json.dump(config_data, f, indent=2)

print("✅ Configuration patched successfully!")
print(f"\nStep 2: Starting MLX conversion from {local_hf_path}...")

Step 1: Patching config.json for MLX compatibility and 3D Vision Math...
✅ Configuration patched successfully!

Step 2: Starting MLX conversion from ./Qwen3VL-2B-Medmaxv2...


In [4]:
from mlx_vlm.convert import convert

# 1. Define paths
local_hf_path = "./Qwen3VL-2B-Medmaxv2"
mlx_output_path = "./Madrimed_vl_2b_mlx"

# NOTE: We are intentionally NOT touching the config.json. 
# The architecture must remain exactly as trained.

print(f"Starting unquantized MLX conversion from {local_hf_path}...")

# Run the conversion directly via the Python API
convert(
    hf_path=local_hf_path,
    mlx_path=mlx_output_path,
    trust_remote_code=True
)

print("\n✅ Model successfully converted to MLX format with full accuracy!")

Starting unquantized MLX conversion from ./Qwen3VL-2B-Medmaxv2...
[INFO] Loading
[INFO] Using dtype: float32

✅ Model successfully converted to MLX format with full accuracy!


In [1]:
import time
from mlx_vlm import load, generate

# 1. Point this to the local folder where you fixed the config.json
MODEL_PATH = "./Madrimed_vl_2b_mlx"

print("Loading model into Apple Unified Memory...")
model, processor = load(MODEL_PATH)


/opt/homebrew/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading model into Apple Unified Memory...


In [4]:

raw_prompt =  """Choose the correct option for the question

Question:
Examine the mammogram image shown above. Which of the following findings is most evident?

Options
A. Well-circumscribed round mass with benign features
B. Clustered microcalcifications within an area of irregular density
C. Fat-containing lesion consistent with lipoma
D. Diffuse bilateral breast edema
"""

image_path = ["MM-1-a.png"]

# --- THE FIX: Format the prompt using the Chat Template ---
messages = [
    {
        "role": "system",
        "content": "You are an expert medical AI. You must deeply analyze the question and provide the final answer."
    },
    {
        "role": "user",
        "content": [
            {"type": "image"},
            {"type": "text", "text": raw_prompt}
        ]
    }
]

# Apply the template so Qwen knows exactly where to put the 65 image features
prompt = processor.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

print("\nStarting generation...")
start_time = time.perf_counter()

# 2. MLX generate function handles the Metal optimization natively
output = generate(
    model, 
    processor, 
    prompt,       # Pass the formatted prompt here!
    image_path, 
    max_tokens=2048,
    verbose=True # Set to True if you want MLX to print its own exact speed stats
)

end_time = time.perf_counter()



Starting generation...
Files: ['MM-1-a.png'] 

Prompt: <|im_start|>system
You are an expert medical AI. You must deeply analyze the question and provide the final answer.<|im_end|>
<|im_start|>user
<|vision_start|><|image_pad|><|vision_end|>Choose the correct option for the question

Question:
Examine the mammogram image shown above. Which of the following findings is most evident?

Options
A. Well-circumscribed round mass with benign features
B. Clustered microcalcifications within an area of irregular density
C. Fat-containing lesion consistent with lipoma
D. Diffuse bilateral breast edema
<|im_end|>
<|im_start|>assistant
<think>

So, let's analyze the mammogram image. The image shows a breast with a heterogeneous density, which is typical for breast tissue. The key feature here is the presence of microcalcifications, which are small calcium deposits that can be seen as white spots on the mammogram. 

Option A mentions a well-circumscribed round mass with benign features. However, t